# Battery analysis: visualization notebook

Reproduce the four requested figures from saved outputs. These are diagnostic high-rate capacity results. RUL uses 80% of cycle-2 measured capacity, assumed BOL. The three LSTM seeds do not define calibrated population uncertainty. Execute top-to-bottom after the README run order, or use the included outputs. No models are retrained here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
ROOT = Path.cwd()
if not (ROOT / "battery_thresholds.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "battery_thresholds.json").exists(), "Open from the repository root or notebooks folder"
DATA = ROOT / "outputs"
def show_figure(name, size=(15, 9)):
    path = DATA / "reporting_visualization" / (name + ".png")
    assert path.exists(), "Run build_task6_visualizations.py first"
    fig, ax = plt.subplots(figsize=size)
    ax.imshow(mpimg.imread(path)); ax.axis("off"); plt.tight_layout(); plt.show()


## 1. Capacity versus cycles
Upper panels are descriptive full-history fits; lower panels are held-out predictions after cycle 85. Future actual capacities are not inserted into recursive forecasting. The square-root-time model is excluded from the predictive panels because its saved evaluation depends on observed future timestamps.
Capacity curves terminate at their first interpolated zero crossing. Raw negative forecasts remain in CSVs and metrics; no flat zero-capacity tail is drawn.

In [ ]:
show_figure("01_capacity_predictive_curves")
metrics = pd.read_csv(DATA / "capacity_comparison_local" / "metrics.csv", dtype={"Dataset_ID": str})
print(metrics[metrics.Stage == "test"].pivot(index="Model", columns="Dataset_ID", values="RMSE_Ah").round(5).to_string())


## 2. Coulombic efficiency
CE = 100 x discharge Ah / charge Ah. Values above 100% remain review flags, not proof of damage. Raw measurements are not clipped or replaced.

In [ ]:
cycle = pd.read_csv(DATA / "capacity_fade" / "cycle_capacity_analysis.csv", dtype={"Dataset_ID": str})
np.testing.assert_allclose(cycle.Coulombic_Efficiency_pct, 100 * cycle.Discharge_Capacity_Ah / cycle.Charge_Capacity_Ah)
print(cycle.loc[cycle.Coulombic_Efficiency_pct > 100, ["Dataset_ID", "Cycle_Index", "Coulombic_Efficiency_pct"]].to_string(index=False))
show_figure("02_coulombic_efficiency", (15, 6))


## 3. Deviation clusters
DBSCAN uses three detrended/scaled features. This chart projects two, so differently labelled points can overlap. Noise means candidate outlier. Cycle 2 is not evaluated. Retrospective centred neighbourhoods are unsuitable as a causal online detector.

In [ ]:
anomaly = pd.read_csv(DATA / "anomaly_detection" / "anomaly_features.csv", dtype={"Dataset_ID": str})
print(anomaly.groupby("Dataset_ID").DBSCAN_Label.apply(lambda s: pd.Series({"evaluated": s.notna().sum(), "noise": (s == -1).sum()})).to_string())
assert int((anomaly.DBSCAN_Label == -1).sum()) == 70
show_figure("03_deviation_clusters")


## 4. Forecasted RUL distribution
All plotted forecasts originate at cycle 8 and train on each cell's own history. LSTM seeds are 42, 7 and 123. Bars count model runs, not independent batteries. True remaining lives are 5 and 16 cycles. Mean EOL values may be fractional only because integer forecasts are averaged.

In [ ]:
samples = pd.read_csv(DATA / "reporting_visualization" / "rul_distribution_samples_checkpoint8.csv", dtype={"Dataset_ID": str})
assert len(samples) == 6 and (samples.Checkpoint_Cycle == 8).all()
np.testing.assert_allclose(samples.Predicted_RUL_Cycles, samples.Predicted_EOL_Cycle - 8)
print(samples[["Dataset_ID", "Seed", "Actual_RUL_Cycles", "Predicted_RUL_Cycles", "Predicted_EOL_Cycle"]].to_string(index=False))
show_figure("04_forecasted_rul_distribution", (15, 6))


## 5. Illustrative pack sensitivity
These results use assumed variation, temperature acceleration and initial SOC-balancing scenarios. They are capacity envelopes, not validated electrothermal simulations or measured pack life. Review outputs/pack_scaling/README.md before interpretation.

In [ ]:
print(pd.read_csv(DATA / "pack_scaling" / "scenario_summary.csv").round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(mpimg.imread(DATA / "pack_scaling" / "scenario_retention.png")); ax.axis("off"); plt.tight_layout(); plt.show()


## Limitations and reporting
Only two cells, distinct loading, absent temperature, unverified reference SOH and very few early training windows. Use the five-page report as the submission summary and README.md for run instructions. Fleet and battery-passport extensions are proposed work, not implemented regulatory compliance.